# Notebook 45 - UltraTrack KLT one-step affine diagnostics

Notebook 44 showed that the oracle-mask OpenCV KLT prototype improves the old KLT artifact, but still drifts badly over the full sequence. This notebook asks a more local question:

> If we reinitialize from MATLAB's previous-frame masks and geometry, can OpenCV track one frame forward and reproduce MATLAB's next pre-Kalman geometry?

That splits the remaining KLT problem into:

1. **Local KLT/affine parity**: one-step point tracking and affine propagation.
2. **Sequential UltraTrack state parity**: keeping the same point set, reinitialization timing, and masks over many frames without drift.

MATLAB did not export raw point tracks, and the saved `affine2d` objects are MATLAB MCOS opaques, so this notebook compares the implied geometry step: apply Python's estimated affine to MATLAB geometry at frame `f-1`, then compare to MATLAB geometry at frame `f`.

In [1]:
from pathlib import Path
import json
import os
import sys
import time

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

import cv2
import numpy as np
import pandas as pd
from scipy.io import loadmat

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultrasound_tracker.geometry import line_angles_batch, line_lengths_batch, normalize_angle
from ultrasound_tracker.matlab_compat import metric_row

OUT = ROOT / "results" / "notebook45_ultratrack_klt_one_step_affine_diagnostics"
OUT.mkdir(parents=True, exist_ok=True)

MATLAB_EXPORT = Path("/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat")
VIDEO_PATH = ROOT / "data" / "raw" / "Test2.mp4"
NB44_NPZ = ROOT / "results" / "notebook44_ultratrack_klt_oracle_mask_gate" / "opencv_ultratrack_klt_oracle_mask_features_arrays.npz"
NB44_METRICS = ROOT / "results" / "notebook44_ultratrack_klt_oracle_mask_gate" / "klt_oracle_mask_full_metrics.csv"

for path in [MATLAB_EXPORT, VIDEO_PATH, NB44_NPZ, NB44_METRICS]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Output folder:", OUT)
print("MATLAB export:", MATLAB_EXPORT)
print("Video:", VIDEO_PATH)

Output folder: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook45_ultratrack_klt_one_step_affine_diagnostics
MATLAB export: /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat
Video: /Users/grosbedou/PycharmProjects/NDORMS/data/raw/Test2.mp4


## Load MATLAB reference geometry

The target for this diagnostic is the MATLAB **pre-Kalman** geometry:

- Fascicle: `Region.Fascicle.fas_x_original` / `fas_y_original`.
- Aponeurosis: `Region.sup_x/sup_y` and `Region.deep_x/deep_y` after MATLAB's aponeurosis affine update.

MATLAB fascicle endpoint order is `[deep; superficial]`; this notebook converts to `[x_sup, y_sup, x_deep, y_deep]`.

In [2]:
mat_root = loadmat(MATLAB_EXPORT, simplify_cells=True)["UTT_numeric_export"]
region = mat_root["Region"]
fascicle = region["Fascicle"]
geofeatures = list(np.asarray(mat_root["geofeatures"], dtype=object).reshape(-1))
parms = mat_root["parms"]

height = int(mat_root["vidHeight"])
width = int(mat_root["vidWidth"])
mat_n = int(mat_root["NumFrames"])
mm_per_px = float(mat_root["ID"]) / height
block_size_matlab = np.asarray(mat_root["BlockSize"], dtype=int).reshape(-1)
contract_win_size = (int(block_size_matlab[1]), int(block_size_matlab[0]))

cap = cv2.VideoCapture(str(VIDEO_PATH))
video_info = {
    "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    "fps": float(cap.get(cv2.CAP_PROP_FPS)),
}
cap.release()

if (video_info["height"], video_info["width"]) != (height, width):
    raise ValueError(f"Video shape mismatch: {video_info} vs {(height, width)}")

print({
    "matlab_frames": mat_n,
    "video_info": video_info,
    "mm_per_px": mm_per_px,
    "matlab_block_size_height_width": block_size_matlab.tolist(),
    "opencv_win_size_width_height": contract_win_size,
})

{'matlab_frames': 2666, 'video_info': {'frame_count': 2667, 'width': 706, 'height': 562, 'fps': 33.341}, 'mm_per_px': 0.09021352313167261, 'matlab_block_size_height_width': [21, 71], 'opencv_win_size_width_height': (71, 21)}


In [3]:
def matlab_pair_series(values: object, n: int | None = None) -> np.ndarray:
    out = []
    for value in np.asarray(values, dtype=object).reshape(-1):
        arr = np.asarray(value, dtype=float).reshape(-1)
        out.append(arr[:2] if arr.size >= 2 else [np.nan, np.nan])
    result = np.asarray(out, dtype=float)
    return result if n is None else result[:n]


def matlab_fascicle_segments(x_field: str, y_field: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(fascicle[x_field], n=n)
    y = matlab_pair_series(fascicle[y_field], n=n)
    return np.column_stack([x[:, 1], y[:, 1], x[:, 0], y[:, 0]])


def matlab_apo_segments(prefix: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(region[f"{prefix}_x"], n=n)
    y = matlab_pair_series(region[f"{prefix}_y"], n=n)
    return np.column_stack([x[:, 0], y[:, 0], x[:, 1], y[:, 1]])


def normalized_angles_deg(segments: np.ndarray) -> np.ndarray:
    return np.asarray([normalize_angle(v, degrees=True) for v in line_angles_batch(segments, degrees=True)], dtype=float)


mat_fascicle = matlab_fascicle_segments("fas_x_original", "fas_y_original", n=mat_n)
mat_sup = matlab_apo_segments("sup", n=mat_n)
mat_deep = matlab_apo_segments("deep", n=mat_n)

print("Reference fascicle frame 0:", mat_fascicle[0])
print("Reference fascicle frame 1:", mat_fascicle[1])
print("Reference superficial apo frame 0:", mat_sup[0])
print("Reference deep apo frame 0:", mat_deep[0])

Reference fascicle frame 0: [733.          54.41875512 -27.         309.01343161]
Reference fascicle frame 1: [732.97476427  54.42288198 -27.01335866 309.01276553]
Reference superficial apo frame 0: [  1.          36.85315315 706.          53.77084357]
Reference deep apo frame 0: [  1.         309.92612613 706.         332.90647011]


## Rebuild MATLAB KLT masks

This uses the same oracle-mask construction as notebook 44:

- Fascicle mask: narrow bands around `geofeatures(f).x/y`, intersected with the Hough-local corrected ROI.
- Aponeurosis mask: superficial and deep cut bands from `parms.apo.super.cut` and `parms.apo.deep.cut`.

For a one-step transition `f-1 -> f`, features are detected in the masks for frame `f-1`, matching the point set MATLAB would use before stepping to frame `f`.

In [4]:
super_cut = np.asarray(parms["apo"]["super"]["cut"], dtype=float).reshape(-1)
deep_cut = np.asarray(parms["apo"]["deep"]["cut"], dtype=float).reshape(-1)


def poly_mask_1b(x_values, y_values, shape=(height, width)) -> np.ndarray:
    x = np.asarray(x_values, dtype=float).reshape(-1) - 1.0
    y = np.asarray(y_values, dtype=float).reshape(-1) - 1.0
    points = np.rint(np.column_stack([x, y])).astype(np.int32)
    mask = np.zeros(shape, dtype=np.uint8)
    cv2.fillPoly(mask, [points], 1)
    return mask.astype(bool)


def tracking_masks(frame_index: int) -> dict[str, np.ndarray]:
    entry = geofeatures[frame_index]

    line_mask = np.zeros((height, width), dtype=bool)
    xs = np.asarray(entry["x"], dtype=float)
    ys = np.asarray(entry["y"], dtype=float)
    for (x1, x2), (y1, y2) in zip(xs, ys):
        px = [x1, x1, x2, x2]
        py = [y1 - 5.0, y1 + 5.0, y2 + 5.0, y2 - 5.0]
        if np.all(np.isfinite(px)) and np.all(np.isfinite(py)):
            line_mask |= poly_mask_1b(px, py)

    super_pos = np.asarray(entry["super_pos"], dtype=float).reshape(-1)
    deep_pos = np.asarray(entry["deep_pos"], dtype=float).reshape(-1)
    thickness = deep_pos - super_pos
    r = 0.1
    roix = [1.0, 1.0, float(width), float(width), 1.0]
    roiy_fcor = np.rint([
        super_pos[0] + thickness[0] * r,
        deep_pos[0] - thickness[0] * r,
        deep_pos[1] - thickness[1] * r,
        super_pos[1] + thickness[1] * r,
        super_pos[0] + thickness[0] * r,
    ])
    fcor_mask = poly_mask_1b(roix, roiy_fcor)
    fascicle_mask = line_mask & fcor_mask

    super_y = [super_cut[0] * height, super_cut[1] * height, super_cut[1] * height, super_cut[0] * height, super_cut[0] * height]
    deep_y = [deep_cut[0] * height, deep_cut[1] * height, deep_cut[1] * height, deep_cut[0] * height, deep_cut[0] * height]
    super_mask = poly_mask_1b(roix, super_y)
    deep_mask = poly_mask_1b(roix, deep_y)

    return {
        "line_mask": line_mask,
        "fcor_mask": fcor_mask,
        "fascicle_mask": fascicle_mask,
        "super_mask": super_mask,
        "deep_mask": deep_mask,
        "apo_mask": super_mask | deep_mask,
    }


def masked_image(gray: np.ndarray, mask: np.ndarray) -> np.ndarray:
    out = np.asarray(gray).copy()
    out[~mask] = 0
    return out

## One-step OpenCV KLT/affine helpers

The local approximation is the same contract used in notebook 44:

- `goodFeaturesToTrack` with `qualityLevel=0.005`, `blockSize=11`.
- Fascicle points capped at 300 strongest points.
- Aponeurosis points uncapped.
- LK `maxLevel=3` for 4 pyramid levels including level 0.
- Full affine RANSAC threshold 50 px.
- Affine estimation is performed in MATLAB-style 1-based coordinates.

In [5]:
def detect_min_eigen_like(gray: np.ndarray, mask: np.ndarray, max_corners: int) -> np.ndarray:
    img = masked_image(gray, mask)
    points = cv2.goodFeaturesToTrack(
        img,
        maxCorners=int(max_corners),
        qualityLevel=0.005,
        minDistance=1,
        blockSize=11,
        useHarrisDetector=False,
    )
    if points is None:
        return np.empty((0, 1, 2), dtype=np.float32)
    return points.astype(np.float32)


def filter_points_by_mask(points: np.ndarray, mask: np.ndarray) -> np.ndarray:
    points = np.asarray(points, dtype=np.float32).reshape(-1, 2)
    if len(points) == 0:
        return points.reshape(0, 1, 2)
    xy = np.rint(points).astype(int)
    xy[:, 0] = np.clip(xy[:, 0], 0, width - 1)
    xy[:, 1] = np.clip(xy[:, 1], 0, height - 1)
    keep = mask[xy[:, 1], xy[:, 0]]
    return points[keep].reshape(-1, 1, 2).astype(np.float32)


def track_points(prev_gray: np.ndarray, gray: np.ndarray, points: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    points = np.asarray(points, dtype=np.float32).reshape(-1, 1, 2)
    if len(points) == 0:
        return np.empty((0, 2), dtype=np.float32), np.empty((0, 2), dtype=np.float32)
    new_points, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_gray,
        gray,
        points,
        None,
        winSize=contract_win_size,
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 50, 0.01),
    )
    if new_points is None or status is None:
        return np.empty((0, 2), dtype=np.float32), np.empty((0, 2), dtype=np.float32)
    keep = status.reshape(-1).astype(bool)
    return points.reshape(-1, 2)[keep], new_points.reshape(-1, 2)[keep]


def estimate_affine_matlab_coords(old_points_0b: np.ndarray, new_points_0b: np.ndarray) -> tuple[np.ndarray | None, int]:
    old_points_0b = np.asarray(old_points_0b, dtype=np.float32).reshape(-1, 2)
    new_points_0b = np.asarray(new_points_0b, dtype=np.float32).reshape(-1, 2)
    if len(old_points_0b) < 3 or len(new_points_0b) < 3:
        return None, 0
    affine, inliers = cv2.estimateAffine2D(
        old_points_0b + 1.0,
        new_points_0b + 1.0,
        method=cv2.RANSAC,
        ransacReprojThreshold=50.0,
        maxIters=2000,
        confidence=0.99,
        refineIters=10,
    )
    if affine is None:
        return None, 0
    n_inliers = int(np.asarray(inliers).sum()) if inliers is not None else len(old_points_0b)
    return affine.astype(np.float32), n_inliers


def apply_affine_1b(segment_1b: np.ndarray, affine: np.ndarray) -> np.ndarray:
    points = np.asarray(segment_1b, dtype=np.float32).reshape(1, -1, 2)
    return cv2.transform(points, affine.astype(np.float32))[0].reshape(-1).astype(float)


def metric(label: str, reference, estimate, unit: str, target_rmse: float | None = None) -> dict:
    row = metric_row(label, reference, estimate)
    row["unit"] = unit
    row["target_rmse"] = target_rmse
    row["passes"] = bool(target_rmse is None or row["rmse"] <= target_rmse)
    return row


def save_metrics(rows: list[dict], path: Path) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    order = ["comparison", "unit", "n", "bias", "mae", "rmse", "corr", "target_rmse", "passes"]
    df = df[[col for col in order if col in df.columns]]
    df.to_csv(path, index=False)
    return df

## Run full one-step diagnostic

For each transition `f-1 -> f`, the notebook:

1. Detects points in frame `f-1` using MATLAB oracle masks for `f-1`.
2. Tracks those points to frame `f`.
3. Fits a full affine in 1-based coordinates.
4. Applies the affine to MATLAB reference geometry at `f-1`.
5. Compares the resulting geometry to MATLAB reference geometry at `f`.

There is no cumulative state in this diagnostic.

In [6]:
def read_next_gray(cap: cv2.VideoCapture) -> np.ndarray:
    ok, frame = cap.read()
    if not ok:
        raise RuntimeError("Could not read next video frame")
    if frame.ndim == 3:
        return cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return frame.copy()


def run_one_step_diagnostic() -> dict:
    out_fascicle = np.full((mat_n, 4), np.nan, dtype=np.float32)
    out_sup = np.full((mat_n, 4), np.nan, dtype=np.float32)
    out_deep = np.full((mat_n, 4), np.nan, dtype=np.float32)
    out_fascicle[0] = mat_fascicle[0]
    out_sup[0] = mat_sup[0]
    out_deep[0] = mat_deep[0]

    f_points_count = np.zeros(mat_n, dtype=np.int32)
    a_points_count = np.zeros(mat_n, dtype=np.int32)
    f_tracked_count = np.zeros(mat_n, dtype=np.int32)
    a_tracked_count = np.zeros(mat_n, dtype=np.int32)
    f_inlier_count = np.zeros(mat_n, dtype=np.int32)
    a_inlier_count = np.zeros(mat_n, dtype=np.int32)
    f_affine_ok = np.zeros(mat_n, dtype=bool)
    a_affine_ok = np.zeros(mat_n, dtype=bool)
    f_affine_matrices = np.full((mat_n, 2, 3), np.nan, dtype=np.float32)
    a_affine_matrices = np.full((mat_n, 2, 3), np.nan, dtype=np.float32)

    cap = cv2.VideoCapture(str(VIDEO_PATH))
    prev_gray = read_next_gray(cap)
    start = time.time()

    for frame in range(1, mat_n):
        gray = read_next_gray(cap)
        m = tracking_masks(frame - 1)

        f_points = detect_min_eigen_like(prev_gray, m["fascicle_mask"], max_corners=300)
        f_points = filter_points_by_mask(f_points, m["fcor_mask"])
        old_f, new_f = track_points(prev_gray, gray, f_points)
        f_affine, f_inliers = estimate_affine_matlab_coords(old_f, new_f)

        a_points = detect_min_eigen_like(prev_gray, m["apo_mask"], max_corners=0)
        a_points = filter_points_by_mask(a_points, m["apo_mask"])
        old_a, new_a = track_points(prev_gray, gray, a_points)
        a_affine, a_inliers = estimate_affine_matlab_coords(old_a, new_a)

        f_points_count[frame] = len(f_points)
        a_points_count[frame] = len(a_points)
        f_tracked_count[frame] = len(new_f)
        a_tracked_count[frame] = len(new_a)

        if f_affine is not None:
            out_fascicle[frame] = apply_affine_1b(mat_fascicle[frame - 1], f_affine)
            f_affine_ok[frame] = True
            f_inlier_count[frame] = f_inliers
            f_affine_matrices[frame] = f_affine

        if a_affine is not None:
            sup_new = apply_affine_1b(mat_sup[frame - 1], a_affine).reshape(2, 2)
            deep_new = apply_affine_1b(mat_deep[frame - 1], a_affine).reshape(2, 2)
            # MATLAB preserves aponeurosis x endpoints as [1, width] after applying the aponeurosis warp.
            out_sup[frame] = [1.0, sup_new[0, 1], float(width), sup_new[1, 1]]
            out_deep[frame] = [1.0, deep_new[0, 1], float(width), deep_new[1, 1]]
            a_affine_ok[frame] = True
            a_inlier_count[frame] = a_inliers
            a_affine_matrices[frame] = a_affine

        prev_gray = gray

        if frame % 500 == 0 or frame == mat_n - 1:
            print(
                f"processed {frame + 1}/{mat_n} frames | "
                f"fpts={f_points_count[frame]} apts={a_points_count[frame]} "
                f"elapsed={time.time() - start:.1f}s"
            )

    cap.release()
    return {
        "frame": np.arange(mat_n, dtype=np.int32),
        "fascicle_segments": out_fascicle,
        "sup_apo_lines": out_sup,
        "deep_apo_lines": out_deep,
        "f_points_count": f_points_count,
        "a_points_count": a_points_count,
        "f_tracked_count": f_tracked_count,
        "a_tracked_count": a_tracked_count,
        "f_inlier_count": f_inlier_count,
        "a_inlier_count": a_inlier_count,
        "f_affine_ok": f_affine_ok,
        "a_affine_ok": a_affine_ok,
        "f_affine_matrices": f_affine_matrices,
        "a_affine_matrices": a_affine_matrices,
        "elapsed_s": time.time() - start,
    }


one_step = run_one_step_diagnostic()

one_step_npz = OUT / "opencv_ultratrack_klt_one_step_affine_arrays.npz"
np.savez(
    one_step_npz,
    frame=one_step["frame"],
    fascicle_segments=one_step["fascicle_segments"],
    sup_apo_lines=one_step["sup_apo_lines"],
    deep_apo_lines=one_step["deep_apo_lines"],
    f_points_count=one_step["f_points_count"],
    a_points_count=one_step["a_points_count"],
    f_tracked_count=one_step["f_tracked_count"],
    a_tracked_count=one_step["a_tracked_count"],
    f_inlier_count=one_step["f_inlier_count"],
    a_inlier_count=one_step["a_inlier_count"],
    f_affine_ok=one_step["f_affine_ok"],
    a_affine_ok=one_step["a_affine_ok"],
    f_affine_matrices=one_step["f_affine_matrices"],
    a_affine_matrices=one_step["a_affine_matrices"],
    win_size=np.asarray(contract_win_size, dtype=np.int32),
    mm_per_px=np.asarray(mm_per_px, dtype=np.float32),
)
print("Saved:", one_step_npz)
print({
    "elapsed_s": one_step["elapsed_s"],
    "f_affine_rate": float(np.mean(one_step["f_affine_ok"][1:])),
    "a_affine_rate": float(np.mean(one_step["a_affine_ok"][1:])),
    "mean_f_points": float(np.mean(one_step["f_points_count"][1:])),
    "mean_a_points": float(np.mean(one_step["a_points_count"][1:])),
    "mean_f_inliers": float(np.mean(one_step["f_inlier_count"][1:])),
    "mean_a_inliers": float(np.mean(one_step["a_inlier_count"][1:])),
})

processed 501/2666 frames | fpts=300 apts=817 elapsed=5.9s


processed 1001/2666 frames | fpts=300 apts=854 elapsed=11.3s


processed 1501/2666 frames | fpts=300 apts=991 elapsed=16.6s


processed 2001/2666 frames | fpts=300 apts=1021 elapsed=22.0s


processed 2501/2666 frames | fpts=300 apts=884 elapsed=27.5s


processed 2666/2666 frames | fpts=300 apts=957 elapsed=29.4s
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook45_ultratrack_klt_one_step_affine_diagnostics/opencv_ultratrack_klt_one_step_affine_arrays.npz
{'elapsed_s': 29.40278196334839, 'f_affine_rate': 1.0, 'a_affine_rate': 1.0, 'mean_f_points': 298.84427767354595, 'mean_a_points': 955.2735459662289, 'mean_f_inliers': 298.83039399624766, 'mean_a_inliers': 955.0131332082551}


## One-step gate metrics

The target here is stricter than the full sequential KLT gate because each step starts from MATLAB geometry. Passing this gate means the local OpenCV affine estimate is good enough; it does **not** mean the sequential KLT state is solved.

In [7]:
valid = np.arange(mat_n) > 0
length_target_px = 2.0 / mm_per_px

est_fas = np.asarray(one_step["fascicle_segments"], dtype=float)
est_sup = np.asarray(one_step["sup_apo_lines"], dtype=float)
est_deep = np.asarray(one_step["deep_apo_lines"], dtype=float)

one_step_rows = [
    metric("one_step_x_sup_px", mat_fascicle[valid, 0], est_fas[valid, 0], "px", target_rmse=2.0),
    metric("one_step_y_sup_px", mat_fascicle[valid, 1], est_fas[valid, 1], "px", target_rmse=2.0),
    metric("one_step_x_deep_px", mat_fascicle[valid, 2], est_fas[valid, 2], "px", target_rmse=2.0),
    metric("one_step_y_deep_px", mat_fascicle[valid, 3], est_fas[valid, 3], "px", target_rmse=2.0),
    metric("one_step_angle_deg", normalized_angles_deg(mat_fascicle[valid]), normalized_angles_deg(est_fas[valid]), "deg", target_rmse=1.0),
    metric("one_step_length_px", line_lengths_batch(mat_fascicle[valid]), line_lengths_batch(est_fas[valid]), "px", target_rmse=length_target_px),
    metric("one_step_length_mm", line_lengths_batch(mat_fascicle[valid]) * mm_per_px, line_lengths_batch(est_fas[valid]) * mm_per_px, "mm", target_rmse=2.0),
    metric("one_step_super_y1_px", mat_sup[valid, 1], est_sup[valid, 1], "px", target_rmse=2.0),
    metric("one_step_super_y2_px", mat_sup[valid, 3], est_sup[valid, 3], "px", target_rmse=2.0),
    metric("one_step_deep_y1_px", mat_deep[valid, 1], est_deep[valid, 1], "px", target_rmse=2.0),
    metric("one_step_deep_y2_px", mat_deep[valid, 3], est_deep[valid, 3], "px", target_rmse=2.0),
]

one_step_metrics = save_metrics(one_step_rows, OUT / "klt_one_step_affine_metrics.csv")
print("Saved:", OUT / "klt_one_step_affine_metrics.csv")
display(one_step_metrics)
print("One-step local KLT/affine gate passes:", bool(one_step_metrics["passes"].all()))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook45_ultratrack_klt_one_step_affine_diagnostics/klt_one_step_affine_metrics.csv


,comparison,unit,n,bias,mae,rmse,corr,target_rmse,passes
0,one_step_x_sup_px,px,2665,-0.006784,0.655737,1.242615,0.994954,2.000000,True
1,one_step_y_sup_px,px,2665,-0.002874,0.169036,0.286703,0.995130,2.000000,True
2,one_step_x_deep_px,px,2665,-0.029243,3.372742,5.725075,0.998618,2.000000,False
3,one_step_y_deep_px,px,2665,-0.011569,0.371991,0.604895,0.992560,2.000000,True
4,one_step_angle_deg,deg,2665,0.001602,0.167112,0.301311,0.998062,1.000000,True
5,one_step_length_px,px,2665,0.026476,3.164069,5.167994,0.998843,22.169625,True
6,one_step_length_mm,mm,2665,0.002388,0.285442,0.466223,0.998843,2.000000,True
7,one_step_super_y1_px,px,2665,-0.001153,0.200973,0.307649,0.990298,2.000000,True
8,one_step_super_y2_px,px,2665,0.000887,0.159647,0.248922,0.992211,2.000000,True
9,one_step_deep_y1_px,px,2665,-0.004070,0.437368,0.727254,0.990102,2.000000,True


One-step local KLT/affine gate passes: False


## Nine validation-frame metrics

These are the same nine focus frames used in the earlier TimTrack gates. The one-step diagnostic for frame `f` uses the transition `f-1 -> f`, so frame 0 is excluded from this table.

In [8]:
validation_frames = np.asarray([122, 533, 691, 955, 1066, 1600, 2133, 2665], dtype=int)

validation_rows = [
    metric("one_step_x_sup_px", mat_fascicle[validation_frames, 0], est_fas[validation_frames, 0], "px", target_rmse=2.0),
    metric("one_step_y_sup_px", mat_fascicle[validation_frames, 1], est_fas[validation_frames, 1], "px", target_rmse=2.0),
    metric("one_step_x_deep_px", mat_fascicle[validation_frames, 2], est_fas[validation_frames, 2], "px", target_rmse=2.0),
    metric("one_step_y_deep_px", mat_fascicle[validation_frames, 3], est_fas[validation_frames, 3], "px", target_rmse=2.0),
    metric("one_step_angle_deg", normalized_angles_deg(mat_fascicle[validation_frames]), normalized_angles_deg(est_fas[validation_frames]), "deg", target_rmse=1.0),
    metric("one_step_length_px", line_lengths_batch(mat_fascicle[validation_frames]), line_lengths_batch(est_fas[validation_frames]), "px", target_rmse=length_target_px),
    metric("one_step_length_mm", line_lengths_batch(mat_fascicle[validation_frames]) * mm_per_px, line_lengths_batch(est_fas[validation_frames]) * mm_per_px, "mm", target_rmse=2.0),
    metric("one_step_super_y1_px", mat_sup[validation_frames, 1], est_sup[validation_frames, 1], "px", target_rmse=2.0),
    metric("one_step_super_y2_px", mat_sup[validation_frames, 3], est_sup[validation_frames, 3], "px", target_rmse=2.0),
    metric("one_step_deep_y1_px", mat_deep[validation_frames, 1], est_deep[validation_frames, 1], "px", target_rmse=2.0),
    metric("one_step_deep_y2_px", mat_deep[validation_frames, 3], est_deep[validation_frames, 3], "px", target_rmse=2.0),
]
validation_metrics = save_metrics(validation_rows, OUT / "klt_one_step_validation_frames_metrics.csv")
print("Saved:", OUT / "klt_one_step_validation_frames_metrics.csv")
display(validation_metrics)
print("One-step validation-frame gate passes:", bool(validation_metrics["passes"].all()))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook45_ultratrack_klt_one_step_affine_diagnostics/klt_one_step_validation_frames_metrics.csv


,comparison,unit,n,bias,mae,rmse,corr,target_rmse,passes
0,one_step_x_sup_px,px,8,-0.128071,1.805434,3.458030,0.959336,2.000000,False
1,one_step_y_sup_px,px,8,0.146276,0.187531,0.320139,0.990156,2.000000,True
2,one_step_x_deep_px,px,8,-4.551693,4.695924,9.252777,0.998754,2.000000,False
3,one_step_y_deep_px,px,8,-0.399071,0.848048,1.730757,0.937589,2.000000,True
4,one_step_angle_deg,deg,8,-0.277959,0.284206,0.562022,0.997306,1.000000,True
5,one_step_length_px,px,8,3.583026,3.881949,8.178804,0.998643,22.169625,True
6,one_step_length_mm,mm,8,0.323237,0.350204,0.737839,0.998643,2.000000,True
7,one_step_super_y1_px,px,8,0.084032,0.137039,0.257832,0.983053,2.000000,True
8,one_step_super_y2_px,px,8,-0.077106,0.147622,0.265682,0.995200,2.000000,True
9,one_step_deep_y1_px,px,8,-0.062673,0.780452,1.368996,0.946143,2.000000,True


One-step validation-frame gate passes: False


## Transition-level drilldown

The transition drilldown makes outliers visible and gives the next implementation notebook concrete frames to inspect.

In [9]:
angle_error = normalized_angles_deg(est_fas) - normalized_angles_deg(mat_fascicle)
length_error_px = line_lengths_batch(est_fas) - line_lengths_batch(mat_fascicle)
endpoint_error = np.sqrt((est_fas - mat_fascicle) ** 2)

transition_df = pd.DataFrame({
    "frame": np.arange(mat_n),
    "angle_error_deg": angle_error,
    "length_error_px": length_error_px,
    "length_error_mm": length_error_px * mm_per_px,
    "x_sup_error_px": est_fas[:, 0] - mat_fascicle[:, 0],
    "y_sup_error_px": est_fas[:, 1] - mat_fascicle[:, 1],
    "x_deep_error_px": est_fas[:, 2] - mat_fascicle[:, 2],
    "y_deep_error_px": est_fas[:, 3] - mat_fascicle[:, 3],
    "max_endpoint_abs_error_px": np.nanmax(np.abs(est_fas - mat_fascicle), axis=1),
    "f_points": one_step["f_points_count"],
    "a_points": one_step["a_points_count"],
    "f_tracked": one_step["f_tracked_count"],
    "a_tracked": one_step["a_tracked_count"],
    "f_inliers": one_step["f_inlier_count"],
    "a_inliers": one_step["a_inlier_count"],
    "f_affine_ok": one_step["f_affine_ok"],
    "a_affine_ok": one_step["a_affine_ok"],
})
transition_df = transition_df.loc[transition_df["frame"] > 0].copy()
transition_path = OUT / "klt_one_step_transition_errors.csv"
transition_df.to_csv(transition_path, index=False)
print("Saved:", transition_path)

display(
    transition_df.assign(abs_angle=lambda df: df["angle_error_deg"].abs(), abs_length=lambda df: df["length_error_px"].abs())
    .sort_values(["max_endpoint_abs_error_px", "abs_angle", "abs_length"], ascending=False)
    .head(12)
)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook45_ultratrack_klt_one_step_affine_diagnostics/klt_one_step_transition_errors.csv


,frame,angle_error_deg,length_error_px,length_error_mm,x_sup_error_px,y_sup_error_px,x_deep_error_px,y_deep_error_px,max_endpoint_abs_error_px,f_points,a_points,f_tracked,a_tracked,f_inliers,a_inliers,f_affine_ok,a_affine_ok,abs_angle,abs_length
1492,1492,-2.453018,42.100637,3.798047,-6.711854,2.148751,-55.059459,2.691156,55.059459,300,1050,300,1041,300,1010,True,True,2.453018,42.100637
1491,1491,-2.603518,33.345462,3.008212,-10.014638,2.196233,-51.369116,-2.581487,51.369116,300,1031,300,1031,293,1010,True,True,2.603518,33.345462
696,696,-1.337030,33.108765,2.986858,-6.219392,-0.544450,-41.548386,3.855033,41.548386,300,1034,300,1034,296,1021,True,True,1.337030,33.108765
157,157,-2.212607,38.377799,3.462196,2.790520,1.465904,-41.213511,0.263132,41.213511,300,1054,300,1054,300,1051,True,True,2.212607,38.377799
637,637,1.843932,-25.015977,-2.256779,3.690453,-1.284102,34.082466,1.464812,34.082466,300,866,300,866,300,831,True,True,1.843932,25.015977
695,695,-1.405085,21.621249,1.950529,-6.429187,1.787336,-31.916717,0.279907,31.916717,300,1012,300,1012,300,1011,True,True,1.405085,21.621249
1225,1225,-1.022974,25.807483,2.328184,-3.808865,-2.000588,-31.176907,2.013650,31.176907,300,1094,300,1094,300,1086,True,True,1.022974,25.807483
2556,2556,-2.017544,26.930850,2.429527,1.079318,-0.093443,-31.109921,0.106506,31.109921,300,1093,300,1093,300,1093,True,True,2.017544,26.930850
1490,1490,-1.756618,23.520042,2.121826,-0.942231,0.583186,-29.647567,-1.667003,29.647567,300,1085,300,1085,300,1085,True,True,1.756618,23.520042
638,638,1.540404,-17.537258,-1.582098,7.086229,-1.982257,29.589031,2.078492,29.589031,300,931,300,921,299,911,True,True,1.540404,17.537258


## Drift decomposition versus notebook 44

If one-step metrics are good but notebook 44 sequential metrics are bad, then local affine estimation is not the primary blocker. The remaining work is matching MATLAB's sequential state: exact point-set refresh, feature-location equivalence, mask updates, and tracker state behavior over time.

In [10]:
nb44 = np.load(NB44_NPZ, allow_pickle=True)
nb44_metrics = pd.read_csv(NB44_METRICS)
nb44_fas = np.asarray(nb44["fascicle_segments"], dtype=float)

comparison_rows = []
for signal, one_name, nb44_name in [
    ("angle_deg", "one_step_angle_deg", "oracle_klt_angle_deg"),
    ("length_px", "one_step_length_px", "oracle_klt_length_px"),
    ("length_mm", "one_step_length_mm", "oracle_klt_length_mm"),
    ("x_sup_px", "one_step_x_sup_px", "oracle_klt_x_sup_px"),
    ("x_deep_px", "one_step_x_deep_px", "oracle_klt_x_deep_px"),
]:
    one_rmse = float(one_step_metrics.loc[one_step_metrics["comparison"] == one_name, "rmse"].iloc[0])
    seq_rmse = float(nb44_metrics.loc[nb44_metrics["comparison"] == nb44_name, "rmse"].iloc[0])
    comparison_rows.append({
        "signal": signal,
        "notebook44_sequential_rmse": seq_rmse,
        "notebook45_one_step_rmse": one_rmse,
        "sequential_minus_one_step_rmse": seq_rmse - one_rmse,
        "one_step_fraction_of_sequential_pct": 100.0 * one_rmse / seq_rmse,
    })

drift_comparison = pd.DataFrame(comparison_rows)
drift_path = OUT / "klt_sequential_vs_one_step_rmse.csv"
drift_comparison.to_csv(drift_path, index=False)
print("Saved:", drift_path)
display(drift_comparison)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook45_ultratrack_klt_one_step_affine_diagnostics/klt_sequential_vs_one_step_rmse.csv


,signal,notebook44_sequential_rmse,notebook45_one_step_rmse,sequential_minus_one_step_rmse,one_step_fraction_of_sequential_pct
0,angle_deg,6.118181,0.301311,5.816870,4.924846
1,length_px,121.015689,5.167994,115.847695,4.270516
2,length_mm,10.917252,0.466223,10.451029,4.270516
3,x_sup_px,15.809669,1.242615,14.567054,7.859841
4,x_deep_px,126.979571,5.725075,121.254496,4.508658


## Readiness decision

This notebook should be read as a gate split:

- If **one-step passes**, the local OpenCV KLT/affine approximation is good enough to package as a helper and the next work is sequential tracker parity.
- If **one-step fails**, the next work is lower still: feature detector equivalence, LK parameters, polygon mask equivalence, or affine estimation behavior.

Either way, Kalman remains a separate downstream gate.

In [11]:
one_step_pass = bool(one_step_metrics["passes"].all())
validation_pass = bool(validation_metrics["passes"].all())

summary = {
    "gate": "UltraTrack/KLT one-step local affine diagnostic",
    "passes_full_one_step_gate": one_step_pass,
    "passes_validation_frame_gate": validation_pass,
    "frames": int(mat_n),
    "transitions": int(mat_n - 1),
    "f_affine_rate": float(np.mean(one_step["f_affine_ok"][1:])),
    "a_affine_rate": float(np.mean(one_step["a_affine_ok"][1:])),
    "mean_f_points": float(np.mean(one_step["f_points_count"][1:])),
    "mean_a_points": float(np.mean(one_step["a_points_count"][1:])),
    "mean_f_inliers": float(np.mean(one_step["f_inlier_count"][1:])),
    "mean_a_inliers": float(np.mean(one_step["a_inlier_count"][1:])),
    "one_step_angle_rmse_deg": float(one_step_metrics.loc[one_step_metrics["comparison"] == "one_step_angle_deg", "rmse"].iloc[0]),
    "one_step_length_rmse_px": float(one_step_metrics.loc[one_step_metrics["comparison"] == "one_step_length_px", "rmse"].iloc[0]),
    "one_step_length_rmse_mm": float(one_step_metrics.loc[one_step_metrics["comparison"] == "one_step_length_mm", "rmse"].iloc[0]),
    "next_action": (
        "Package one-step KLT helpers and implement sequential MATLAB-style point refresh/state behavior."
        if one_step_pass
        else "Inspect feature detector, LK, mask, and affine outliers before packaging."
    ),
}
summary_path = OUT / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved:", summary_path)
print(json.dumps(summary, indent=2))

readiness = pd.DataFrame([
    {
        "gate": "TimTrack mask/doHough",
        "status": "passed",
        "ready_for_next": True,
    },
    {
        "gate": "UltraTrack/KLT local one-step affine",
        "status": "passed" if one_step_pass else "not passed",
        "ready_for_next": one_step_pass,
    },
    {
        "gate": "UltraTrack/KLT sequential state",
        "status": "not passed in notebook 44",
        "ready_for_next": False,
    },
    {
        "gate": "MATLAB 2-state Kalman",
        "status": "not started; keep separate",
        "ready_for_next": False,
    },
])
readiness_path = OUT / "readiness.csv"
readiness.to_csv(readiness_path, index=False)
print("Saved:", readiness_path)
display(readiness)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook45_ultratrack_klt_one_step_affine_diagnostics/summary.json
{
  "gate": "UltraTrack/KLT one-step local affine diagnostic",
  "passes_full_one_step_gate": false,
  "passes_validation_frame_gate": false,
  "frames": 2666,
  "transitions": 2665,
  "f_affine_rate": 1.0,
  "a_affine_rate": 1.0,
  "mean_f_points": 298.84427767354595,
  "mean_a_points": 955.2735459662289,
  "mean_f_inliers": 298.83039399624766,
  "mean_a_inliers": 955.0131332082551,
  "one_step_angle_rmse_deg": 0.3013109932225762,
  "one_step_length_rmse_px": 5.167993935651766,
  "one_step_length_rmse_mm": 0.46622294045826423,
  "next_action": "Inspect feature detector, LK, mask, and affine outliers before packaging."
}
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook45_ultratrack_klt_one_step_affine_diagnostics/readiness.csv


,gate,status,ready_for_next
0,TimTrack mask/doHough,passed,True
1,UltraTrack/KLT local one-step affine,not passed,False
2,UltraTrack/KLT sequential state,not passed in notebook 44,False
3,MATLAB 2-state Kalman,not started; keep separate,False
